In [1]:
pip install pandas statsmodels pingouin

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd

# differences average length of context files

csv_files = [
    "contexts_wikipedia.csv", 
    "contexts_conspiracy.csv",
    "contexts_isot_true.csv",
    "contexts_isot_fake.csv"
]
results = []

for file in csv_files:
    data = pd.read_csv(file, usecols=['text_length'])
    
    mean_length = data['text_length'].mean()
    variance_length = data['text_length'].var()
    
    results.append({
        'file': file,
        'mean length': mean_length,
        'var': variance_length
    })
        

results_df = pd.DataFrame(results)
print(results_df)


                      file  mean length           var
0   contexts_wikipedia.csv   405.292829  24581.265662
1  contexts_conspiracy.csv   411.630000  26441.431958
2   contexts_isot_true.csv   528.432750  18533.030235
3   contexts_isot_fake.csv   499.597000  33663.808543


In [ ]:
import pandas as pd

data = pd.read_csv("contexts_conspiracy.csv")
data['text_length'] = data['context_text'].astype(str).apply(len)
data.to_csv("contexts_conspiracy.csv", index=False)

In [ ]:
import pandas as pd

texts_df = pd.read_csv("contexts_isot_fake.csv")
results_df = pd.read_csv("results_isot_fake.csv")

texts_unique = texts_df[["context_title", "text_length"]].drop_duplicates(subset="context_title")

merged_df = results_df.merge(texts_unique, on="context_title", how="left", suffixes=("", "_new"))

merged_df["text_length"] = merged_df["text_length"].combine_first(merged_df["text_length_new"])
merged_df["text_length"] = merged_df["text_length"].astype(int)

merged_df.drop(columns=["text_length_new"], inplace=True)
merged_df.to_csv("results_isot_fake.csv", index=False)



In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
import numpy as np

# subset with similar lengths, wiki vs consp, clean only

wiki_results = pd.read_csv("results_wikipedia.csv")
cons_results = pd.read_csv("results_conspiracy.csv")

wiki_results = wiki_results[wiki_results['context_type'] == 'clean'].copy()
cons_results = cons_results[cons_results['context_type'] == 'clean'].copy()

print(len(wiki_results))

min_tokens = 400
max_tokens = 500

wiki_subset = wiki_results[(wiki_results['text_length'] >= min_tokens) & (wiki_results['text_length'] <= max_tokens)].copy()
cons_subset = cons_results[(cons_results['text_length'] >= min_tokens) & (cons_results['text_length'] <= max_tokens)].copy()

print(len(wiki_subset))
print(len(cons_subset))

wiki_titles = wiki_subset['context_title'].unique()
cons_titles = cons_subset['context_title'].unique()

min_titles = min(len(wiki_titles), len(cons_titles))

np.random.seed(28)
wiki_titles_sampled = np.random.choice(wiki_titles, min_titles, replace=False)
cons_titles_sampled = np.random.choice(cons_titles, min_titles, replace=False)

wiki_sample = wiki_subset[wiki_subset['context_title'].isin(wiki_titles_sampled)].copy()
cons_sample = cons_subset[cons_subset['context_title'].isin(cons_titles_sampled)].copy()

print(len(wiki_sample))
print(len(cons_sample))
wiki_sample['content'] = 'wikipedia'
cons_sample['content'] = 'conspiracy'

data = pd.concat([wiki_sample, cons_sample], ignore_index=True)
data['content'] = data['content'].astype('category')
data['content'] = data['content'].cat.set_categories(['wikipedia','conspiracy'], ordered=True)

model = smf.mixedlm("accuracy ~ content", data, groups=data["model_name"])
result = model.fit(reml=False)
print(result.summary())

coef = result.params.get('content[T.conspiracy]', None)
pval = result.pvalues.get('content[T.conspiracy]', None)
if coef is not None:
    print(f"Wikipedia - conspiracy (length 400–500):")
    print(f"Δ accuracy = {coef:.4f}, p-value = {pval:.4g}")



4016
592
640
592
592
             Mixed Linear Model Regression Results
Model:                MixedLM   Dependent Variable:   accuracy 
No. Observations:     1184      Method:               ML       
No. Groups:           4         Scale:                0.0022   
Min. group size:      296       Log-Likelihood:       1943.3380
Max. group size:      296       Converged:            Yes      
Mean group size:      296.0                                    
---------------------------------------------------------------
                      Coef. Std.Err.   z    P>|z| [0.025 0.975]
---------------------------------------------------------------
Intercept             0.509    0.026 19.446 0.000  0.458  0.561
content[T.conspiracy] 0.021    0.003  7.892 0.000  0.016  0.027
Group Var             0.003    0.042                           


Wikipedia - conspiracy (length 400–500, balanced):
Δ accuracy = 0.0213, p-value = 2.965e-15


/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
import numpy as np

# wiki vs conspiracy, clean only, no length adjustment

wiki_results = pd.read_csv("results_wikipedia.csv")
cons_results = pd.read_csv("results_conspiracy.csv")

wiki_results = wiki_results[wiki_results['context_type'] == 'clean'].copy()
cons_results = cons_results[cons_results['context_type'] == 'clean'].copy()

wiki_results['content'] = 'wikipedia'
cons_results['content'] = 'conspiracy'

data = pd.concat([wiki_results, cons_results], ignore_index=True)

data['content'] = data['content'].astype('category')
data['content'] = data['content'].cat.set_categories(['wikipedia','conspiracy'], ordered=True)

model = smf.mixedlm("accuracy ~ content", data, groups=data["model_name"])
result = model.fit(reml=False)
print(result.summary())

coef = result.params.get('content[T.conspiracy]', None)
pval = result.pvalues.get('content[T.conspiracy]', None)
if coef is not None:
    print(f"Wikipedia - conspiracy:")
    print(f"Δ accuracy = {coef:.4f}, p-value = {pval:.4g}")




             Mixed Linear Model Regression Results
Model:               MixedLM   Dependent Variable:   accuracy  
No. Observations:    8016      Method:               ML        
No. Groups:          4         Scale:                0.0017    
Min. group size:     2004      Log-Likelihood:       14219.1408
Max. group size:     2004      Converged:            Yes       
Mean group size:     2004.0                                    
---------------------------------------------------------------
                      Coef. Std.Err.   z    P>|z| [0.025 0.975]
---------------------------------------------------------------
Intercept             0.513    0.024 21.043 0.000  0.465  0.561
content[T.conspiracy] 0.016    0.001 17.436 0.000  0.014  0.018
Group Var             0.002    0.040                           


Wikipedia - conspiracy:
Δ accuracy = 0.0160, p-value = 4.391e-68


/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

# Wiki vs conspiracy, clean only, with length

wiki_results = pd.read_csv("results_wikipedia.csv")
cons_results = pd.read_csv("results_conspiracy.csv")

wiki_results = wiki_results[wiki_results['context_type'] == 'clean'].copy()
cons_results = cons_results[cons_results['context_type'] == 'clean'].copy()

wiki_results['content'] = 'wikipedia'
cons_results['content'] = 'conspiracy'

data = pd.concat([wiki_results, cons_results], ignore_index=True)

data['content'] = data['content'].astype('category')
data['content'] = data['content'].cat.set_categories(
    ['wikipedia', 'conspiracy'],
    ordered=True
)

# fit LMM
# - fixed effects: content, text_length
# - random intercepts: model_name, context_title
model = smf.mixedlm(
    "accuracy ~ content + scale(text_length)",
    data,
    groups="model_name",
    vc_formula={
        "context": "0 + C(context_title)"
    }
)

result = model.fit(reml=False)
print(result.summary())

coef = result.params.get('content[T.conspiracy]', None)
pval = result.pvalues.get('content[T.conspiracy]', None)

if coef is not None:
    print("Wikipedia vs. conspiracy (clean texts length-adjusted):")
    print(f"Δ accuracy = {coef:.4f}")
    print(f"p-value = {pval:.4g}")


/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)


              Mixed Linear Model Regression Results
Model:               MixedLM    Dependent Variable:    accuracy  
No. Observations:    8016       Method:                ML        
No. Groups:          4          Scale:                 0.0021    
Min. group size:     2004       Log-Likelihood:        10652.4591
Max. group size:     2004       Converged:             Yes       
Mean group size:     2004.0                                      
-----------------------------------------------------------------
                      Coef.  Std.Err.    z    P>|z| [0.025 0.975]
-----------------------------------------------------------------
Intercept              0.513    0.001 741.300 0.000  0.512  0.515
content[T.conspiracy]  0.016    0.001  11.157 0.000  0.013  0.019
scale(text_length)    -0.000    0.001  -0.100 0.920 -0.001  0.001
context Var            0.002                                     


Wikipedia vs. conspiracy (clean texts, length-adjusted):
Δ accuracy = 0.0160
p-value    

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import numpy as np

# Qwen only, four different shuffles, with length, wiki and conspi
###################################################

true = pd.read_csv("results_wikipedia.csv")
fake = pd.read_csv("results_conspiracy.csv")

true = true[true["model_name"].str.contains("Qwen")].copy()
fake = fake[fake["model_name"].str.contains("Qwen")].copy()

true["content"] = "wikipedia"
fake["content"] = "conspiracy"

data = pd.concat([true, fake], ignore_index=True)

data = data[data["context_type"].isin(["clean", "meani", "wordd", "chara"])]

data["content"] = data["content"].astype("category")
data["content"] = data["content"].cat.set_categories(["wikipedia", "conspiracy"], ordered=True)

data["context_type"] = data["context_type"].astype("category")
data["context_type"] = data["context_type"].cat.set_categories(
    ["clean", "meani", "wordd", "chara"],
    ordered=True
)

model = smf.mixedlm(
    "accuracy ~ content * context_type + scale(text_length)",
    data,
    groups="context_title"
)

result = model.fit(reml=True)
print(result.summary())


/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


                         Mixed Linear Model Regression Results
Model:                       MixedLM           Dependent Variable:           accuracy  
No. Observations:            8016              Method:                       REML      
No. Groups:                  2004              Scale:                        0.0003    
Min. group size:             4                 Log-Likelihood:               19532.2370
Max. group size:             4                 Converged:                    Yes       
Mean group size:             4.0                                                       
---------------------------------------------------------------------------------------
                                            Coef.  Std.Err.    z    P>|z| [0.025 0.975]
---------------------------------------------------------------------------------------
Intercept                                    0.563    0.001 786.409 0.000  0.562  0.564
content[T.conspiracy]                        0.017    0.0

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

# wiki vs conspi, clean and meani, with length, reml true
##############################################################

wiki_results = pd.read_csv("results_wikipedia.csv")
cons_results = pd.read_csv("results_conspiracy.csv")

wiki_results = wiki_results[wiki_results['context_type'].isin(['clean', 'meani'])].copy()
cons_results = cons_results[cons_results['context_type'].isin(['clean', 'meani'])].copy()

wiki_results['content'] = 'wikipedia'
cons_results['content'] = 'conspiracy'

data = pd.concat([wiki_results, cons_results], ignore_index=True)

data['content'] = data['content'].astype('category')
data['content'] = data['content'].cat.set_categories(['wikipedia', 'conspiracy'], ordered=True)

data['context_type'] = data['context_type'].astype('category')
data['context_type'] = data['context_type'].cat.set_categories(['clean', 'meani'], ordered=True)

# fit lmm
# fixed effects: content, context_type, interaction, text_length
# random: model_name, context_title
model = smf.mixedlm(
    "accuracy ~ content * context_type + scale(text_length)",
    data,
    groups="model_name",
    vc_formula={"context": "0 + C(context_title)"}
)

result = model.fit(reml=True)
print(result.summary())



/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


                         Mixed Linear Model Regression Results
Model:                       MixedLM           Dependent Variable:           accuracy  
No. Observations:            16032             Method:                       REML      
No. Groups:                  4                 Scale:                        0.0006    
Min. group size:             4008              Log-Likelihood:               26550.0690
Max. group size:             4008              Converged:                    Yes       
Mean group size:             4008.0                                                    
---------------------------------------------------------------------------------------
                                            Coef.  Std.Err.    z    P>|z| [0.025 0.975]
---------------------------------------------------------------------------------------
Intercept                                    0.513    0.001 509.712 0.000  0.511  0.515
content[T.conspiracy]                        0.016    0.0

KeyError: 'Δ accuracy content (conspiracy vs wiki)'

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

# true vs fake, clean and meani, with length

isot_true_results = pd.read_csv("results_isot_true_temporally.csv")
isot_fake_results = pd.read_csv("results_isot_fake.csv")

isot_true_results = isot_true_results[isot_true_results['context_type'].isin(['clean', 'meani'])].copy()
isot_fake_results = isot_fake_results[isot_fake_results['context_type'].isin(['clean', 'meani'])].copy()

isot_true_results['content'] = 'isot_true'
isot_fake_results['content'] = 'isot_fake'

data = pd.concat([isot_true_results, isot_fake_results], ignore_index=True)

data['content'] = data['content'].astype('category')
data['content'] = data['content'].cat.set_categories(['isot_true', 'isot_fake'], ordered=True)

data['context_type'] = data['context_type'].astype('category')
data['context_type'] = data['context_type'].cat.set_categories(['clean', 'meani'], ordered=True)

# fit lmm
# Fixed effects: content, context_type, interaction, text_length (scaled)
# random: model_name, context_title
model = smf.mixedlm(
    "accuracy ~ content * context_type + scale(text_length)",
    data,
    groups="model_name",
    vc_formula={"context": "0 + C(context_title)"}
)

result = model.fit(reml=True)
print(result.summary())


/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


                        Mixed Linear Model Regression Results
Model:                      MixedLM           Dependent Variable:           accuracy  
No. Observations:           15770             Method:                       REML      
No. Groups:                 4                 Scale:                        0.0008    
Min. group size:            3811              Log-Likelihood:               24710.4163
Max. group size:            4000              Converged:                    Yes       
Mean group size:            3942.5                                                    
--------------------------------------------------------------------------------------
                                           Coef.  Std.Err.    z    P>|z| [0.025 0.975]
--------------------------------------------------------------------------------------
Intercept                                   0.539    0.001 521.330 0.000  0.537  0.541
content[T.isot_fake]                       -0.007    0.001  -4.791 0

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

# wiki vs conspiracy, clean and meani, original 
############################################################################

wiki_results = pd.read_csv("results_wikipedia.csv")
cons_results = pd.read_csv("results_conspiracy.csv")

wiki_results = wiki_results[wiki_results['context_type'].isin(['clean', 'meani'])].copy()
cons_results = cons_results[cons_results['context_type'].isin(['clean', 'meani'])].copy()

wiki_results['content'] = 'wikipedia'
cons_results['content'] = 'conspiracy'

data = pd.concat([wiki_results, cons_results], ignore_index=True)

data['content'] = (
    data['content']
    .astype('category')
    .cat.set_categories(['wikipedia', 'conspiracy'], ordered=True)
)

data['context_type'] = (
    data['context_type']
    .astype('category')
    .cat.set_categories(['clean', 'meani'], ordered=True)
)

############################################

model_original = smf.mixedlm(
    "accuracy ~ content * context_type + scale(text_length)",
    data,
    groups="model_name",
    vc_formula={"context": "0 + C(context_title)"}
)

result_original = model_original.fit(reml=True, method="lbfgs")

print("random intercepts only:")
print(result_original.summary())



/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


random intercepts only:
                         Mixed Linear Model Regression Results
Model:                       MixedLM           Dependent Variable:           accuracy  
No. Observations:            16032             Method:                       REML      
No. Groups:                  4                 Scale:                        0.0006    
Min. group size:             4008              Log-Likelihood:               26550.0690
Max. group size:             4008              Converged:                    Yes       
Mean group size:             4008.0                                                    
---------------------------------------------------------------------------------------
                                            Coef.  Std.Err.    z    P>|z| [0.025 0.975]
---------------------------------------------------------------------------------------
Intercept                                    0.513    0.001 509.717 0.000  0.511  0.515
content[T.conspiracy]            

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf

# wiki vs conspiracy, clean and meani, original
############################################################################

wiki_results = pd.read_csv("results_wikipedia.csv")
cons_results = pd.read_csv("results_conspiracy.csv")

wiki_results = wiki_results[wiki_results['context_type'].isin(['clean', 'meani'])].copy()
cons_results = cons_results[cons_results['context_type'].isin(['clean', 'meani'])].copy()

wiki_results['content'] = 'wikipedia'
cons_results['content'] = 'conspiracy'

data = pd.concat([wiki_results, cons_results], ignore_index=True)

data['content'] = (
    data['content']
    .astype('category')
    .cat.set_categories(['wikipedia', 'conspiracy'], ordered=True)
)

data['context_type'] = (
    data['context_type']
    .astype('category')
    .cat.set_categories(['clean', 'meani'], ordered=True)
)

data['text_length_s'] = (data['text_length'] - data['text_length'].mean()) / data['text_length'].std()

############################################

model_original = smf.mixedlm(
    "accuracy ~ content * context_type + text_length_s",
    data,
    groups="model_name",
    vc_formula={"context": "0 + C(context_title)"}
)

result_original = model_original.fit(reml=True, method="lbfgs")

print("random intercepts only:")
print(result_original.summary())




/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


random intercepts only:
                         Mixed Linear Model Regression Results
Model:                       MixedLM           Dependent Variable:           accuracy  
No. Observations:            16032             Method:                       REML      
No. Groups:                  4                 Scale:                        0.0006    
Min. group size:             4008              Log-Likelihood:               26550.0690
Max. group size:             4008              Converged:                    Yes       
Mean group size:             4008.0                                                    
---------------------------------------------------------------------------------------
                                            Coef.  Std.Err.    z    P>|z| [0.025 0.975]
---------------------------------------------------------------------------------------
Intercept                                    0.513    0.001 509.717 0.000  0.511  0.515
content[T.conspiracy]            

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import numpy as np

# Qwen only, four different shuffles, with length, true and fake ---> added slopes for different shuffles
###################################################

true = pd.read_csv("results_isot_true.csv")
fake = pd.read_csv("results_isot_fake.csv")

true = true[true["model_name"].str.contains("Qwen")].copy()
fake = fake[fake["model_name"].str.contains("Qwen")].copy()

true["content"] = "isot_true"
fake["content"] = "isot_fake"

data = pd.concat([true, fake], ignore_index=True)

data = data[data["context_type"].isin(["clean", "meani", "wordd", "chara"])]

data["content"] = data["content"].astype("category")
data["content"] = data["content"].cat.set_categories(["isot_true", "isot_fake"], ordered=True)

data["context_type"] = data["context_type"].astype("category")
data["context_type"] = data["context_type"].cat.set_categories(["clean", "meani", "wordd", "chara"], ordered=True)

model = smf.mixedlm(
    "accuracy ~ content * context_type + scale(text_length)",
    data,
    groups="context_title",
    re_formula="~context_type"
)

result = model.fit(reml=True)
print(result.summary())


/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2200: ConvergenceWarning: Retrying MixedLM optimization with lbfgs
  warnings.warn(
/scratch/fast/noopas/llm_env/lib/python3.9/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


                            Mixed Linear Model Regression Results
Model:                          MixedLM             Dependent Variable:             accuracy  
No. Observations:               8000                Method:                         REML      
No. Groups:                     2000                Scale:                          0.0001    
Min. group size:                4                   Log-Likelihood:                 21396.9135
Max. group size:                4                   Converged:                      Yes       
Mean group size:                4.0                                                           
----------------------------------------------------------------------------------------------
                                                  Coef.  Std.Err.    z     P>|z| [0.025 0.975]
----------------------------------------------------------------------------------------------
Intercept                                          0.603    0.001 1021.344 0.00